In [ ]:
![ -d /kaggle/working/codapath/.git ] || git clone https://github.com/CryAndRRich/codapath.git /kaggle/working/codapath

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
!pip install -r requirements.txt
!pip install -U huggingface_hub hf-transfer

In [ ]:
import os
from huggingface_hub import snapshot_download

# DINOv2 is public; Kaggle Internet must be enabled.
print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import numpy as np
import torch

from set_up import set_seed
from load_data import get_data_loaders
from model import DINOv2Extractor, extract_image_features
from trainer import load_model
from evaluate import evaluate_model

In [ ]:
PATHMNIST_PATH  = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH   = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
CONFIG_PATH = "config/config.yaml"

# pathmnist | histoset | skintissue
DATASET = "histoset"

# Use the saved output prefix. For nucleus_al this defaults to, e.g.,
# nucleus_cellvit_embedding_disagreement.
RUN_NAME = "random"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

random_seed = config["random_seed"]
device = torch.device(config["device"])

In [ ]:
set_seed(random_seed)

_, test_loader, class_names = get_data_loaders(DATA_DICT[DATASET], random_seed, verbose=True)
test_dataset = test_loader.dataset
test_labels = (
    test_dataset.lbl
    if hasattr(test_dataset, "lbl")
    else np.array(test_dataset.dataset.targets)[test_dataset.indices]
)

vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
extractor = DINOv2Extractor(model_name=vit_name).to(device)
test_features = extract_image_features(test_loader, extractor, device)
del extractor

In [ ]:
for budget in config["cumulative_budget"]:
    checkpoint_file = (
        f"{CODAPATH}/checkpoints/{DATASET}/"
        f"{RUN_NAME}_probe_budget_{budget}.pt"
    )

    probe = load_model(checkpoint_file, device)

    print(f"=== {DATASET} | {RUN_NAME} | budget={budget} ===")
    evaluate_model(probe, test_features, test_labels, device)
    print()